In [1]:
pip install opencv-python numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import cv2
import numpy as np

# Load images
img1 = cv2.imread("rgb.jpg", 0)
img2 = cv2.imread("ms.png", 0)

# Create ORB detector
orb = cv2.ORB_create(nfeatures=1000)

# Detect keypoints and descriptors
kp1, des1 = orb.detectAndCompute(img1, None)
kp2, des2 = orb.detectAndCompute(img2, None)

# Create brute force matcher
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

# Match descriptors
matches = bf.match(des1, des2)

# Sort matches by distance
matches = sorted(matches, key=lambda x: x.distance)

# Keep top matches
good_matches = matches[:100]

# Extract matched points
src_pts = np.float32(
    [kp1[m.queryIdx].pt for m in good_matches]
).reshape(-1, 1, 2)

dst_pts = np.float32(
    [kp2[m.trainIdx].pt for m in good_matches]
).reshape(-1, 1, 2)

# Compute homography using RANSAC
H, mask = cv2.findHomography(
    src_pts,
    dst_pts,
    cv2.RANSAC,
    5.0
)

# Count inliers
inliers = int(np.sum(mask))
total_matches = len(good_matches)

# Inlier ratio
inlier_ratio = inliers / total_matches

# Print results
print("\n===== ORB RESULTS =====")
print("Total Matches:", total_matches)
print("Inliers:", inliers)
print("Inlier Ratio:", round(inlier_ratio, 3))

# Draw matches
matched_img = cv2.drawMatches(
    img1,
    kp1,
    img2,
    kp2,
    good_matches,
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

# Save output image
cv2.imwrite("orb_matches.jpg", matched_img)

print("Saved: orb_matches.jpg")


===== ORB RESULTS =====
Total Matches: 100
Inliers: 7
Inlier Ratio: 0.07
Saved: orb_matches.jpg
